<p style="padding: 10px; border: 1px solid black;">
<img src="../common/images/mlu-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-image: linear-gradient(145deg, rgba(35, 47, 62, 1) 0%, rgba(51, 0, 102, 1) 40%, rgba(223, 42, 93, 1) 60%, rgba(124, 90, 237, 1) 85%, rgba(124, 232, 244, 1) 100%); padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Application of Deep Learning to Text and Image Data</h1>
    <h2 style="color: white; margin-top: 15px;">Module 2, Lab 5: Finetuning BERT</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../common/images/mlu-activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../common/images/mlu-challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #f2f0fc; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2f1381; border-bottom: 1px solid #2f1381; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2f1381; font-weight: bold; text-decoration: none;">1. Reading and formatting the dataset</a></p>
    <p><a href="#section2" style="color: #2f1381; font-weight: bold; text-decoration: none;">2. Loading the pre-trained model</a></p>
    <p><a href="#section3" style="color: #2f1381; font-weight: bold; text-decoration: none;">3. Training and testing the model</a></p>
    <ul style="margin-top: 0; padding-left: 30px;">
        <li><a href="#section3-1" style="color: #2f1381; text-decoration: none;">3.1 Looking at what's going on</a></li>
    </ul>
    <p><a href="#section4" style="color: #2f1381; font-weight: bold; text-decoration: none;">4. Getting predictions on the test data</a></p>
</div>

<!-- Important Note -->
<div style="background-color: #FDEDEC; border-left: 5px solid #E74C3C; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="color: #C0392B; margin: 0;"><strong>Important:</strong> BERT and its variants use more resources than the other models you have used so far. This may cause your instance to run out of memory. If that happens, restart the kernel (Kernel->Restart from the top menu), reduce the batch size, then re-run the code.</p>
</div>

<!-- Tip Box -->
<div style="background-color: #E8F8F5; border-left: 5px solid #1ABC9C; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="color: #16A085; margin: 0;"><strong>Tip:</strong> In this walkthrough, you will use a light version of the original BERT implementation called "DistilBert". You can checkout <a href="https://arxiv.org/pdf/1910.01108.pdf">the paper</a> about it for more details.</p>
</div>

In [1]:
# Remove conflicting packages that are not used by this notebook
!pip uninstall -y -q fastai autogluon-multimodal autogluon-timeseries torchvision torchtext timm tensorflow 2>/dev/null || true
!pip install -U -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker-serve 1.12.0 requires onnxruntime, which is not installed.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 25.0.1 which is incompatible.
autogluon-core 1.5.0 requires matplotlib<3.11,>=3.7.0, but you have matplotlib 3.11.1 which is incompatible.
awswrangler 3.17.0 requires pyarrow<25.0.0,>=8.0.0, but you have pyarrow 25.0.1 which is incompatible.
mlflow 3.13.0 requires pyarrow<25,>=4.0.0, but you have pyarrow 25.0.1 which is incompatible.
sagemaker-studio 1.1.22 requires pandas<3.0.0,>=2.3.2, but you have pandas 2.2.3 which is incompatible.


<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">1. Reading and formatting the dataset</h2>
</div>

First, you need to read in the product review dataset and prepare it for the BERT model. To keep the training time down, you will only use the first 2000 data points from the dataset. If you want to improve your model after you understand how to train, you can use more data to train a new model.

This lab uses a dataset derived from a small sample of Amazon product reviews.

__Review dataset schema:__
* __reviewText:__ Text of the review
* __summary:__ Summary of the review
* __verified:__ Whether the purchase was verified (True or False)
* __time:__ UNIX timestamp for the review
* __log\_votes:__ Logarithm-adjusted votes log(1+votes)
* __isPositive:__ Whether the review is positive or negative (1 or 0)

In [2]:
import os
import sys
import time
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from torch.utils.data import DataLoader

# Import system library and append path
sys.path.insert(1, '..')

# Setting tokenizer parallelism to false
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Import utility functions that provide answers to challenges
from MLUDTI_EN_M2_Lab5_quiz_questions import *

Read the dataset.

In [3]:
df = pd.read_csv("../data/NLP-REVIEW-DATA-CLASSIFICATION-TRAINING.csv")

Print the dataset information to see the field types.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56000 entries, 0 to 55999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          56000 non-null  int64  
 1   reviewText  55989 non-null  object 
 2   summary     55987 non-null  object 
 3   verified    56000 non-null  bool   
 4   time        56000 non-null  int64  
 5   log_votes   56000 non-null  float64
 6   isPositive  56000 non-null  int64  
dtypes: bool(1), float64(1), int64(3), object(2)
memory usage: 2.6+ MB


You do not need any of the rows that do not have __reviewText__, so drop them.

In [5]:
df.dropna(subset=["reviewText"], inplace=True)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Understanding Epochs and Learning Rate</h4>
        <p>Answer the question below to test your understanding of epochs and learning rate.</p>
    </div>
</div>

In [6]:
question_1

BERT requires a lot of compute power for large datasets. To reduce the amount of time it takes to train the model, you will only use the first 2,000 data points for this lab. 

In [7]:
df = df.head(2000)

Now split the dataset into training and validation data sets, keeping 10% of the data for validation.

In [8]:
# This separates 10% of the entire dataset into validation dataset.
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["reviewText"].tolist(),
    df["isPositive"].tolist(),
    test_size=0.10,
    shuffle=True,
    random_state=324,
    stratify = df["isPositive"].tolist(),
)

You need to tokenize the data. To do this, use a special tokenizer built for the DistilBERT model to tokenize the training and validation texts. 

In [9]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts,
                            truncation=True,
                            padding=True)
val_encodings = tokenizer(val_texts,
                          truncation=True,
                          padding=True)

Create a new `ReviewDataset` class to use with the BERT model. Later, you use the training and validation encoding-label pairs with this new class.

In [10]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]).to(device) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx]).to(device)
        return item

    def __len__(self):
        return len(self.labels)
    
train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">2. Loading the pre-trained model</h2>
</div>

Now, you need to load the model. When you do this, several warnings will print that are related to the last classification layer of BERT where you are using a randomly initialized layer. You can ignore the warnings as they are not relevant to the type of training you are doing.

In [11]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                            num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


The last step is to freeze all weights until the very last classification layer in the BERT model. This helps accelerate the training process. Training the weights of the whole network (66 million weights) takes a long time. Additionally, 2000 data points would not be enough for that task. Instead, the code below freezes all the weights until the last classification layer. This means only a small portion of the weights gets updated (rest stays the same). This is a common practice with large language models.

In [12]:
# Freeze the encoder weights until the classifier
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">3. Training and testing the model</h2>
</div>

Now that your data is ready and you have configured your model, its time to start the fine-tuning process. This code will take __a long time__ (30+ minutes) to complete with large datasets, that is why you are running it on a subset of the full review dataset.

First, define the accuracy function.

In [13]:
def calculate_accuracy(output, label):
    """Calculate the accuracy of the trained network. 
    output: (batch_size, num_output) float32 tensor
    label: (batch_size, ) int32 tensor """
    
    return (output.argmax(axis=1) == label.float()).float().mean()

Now you need to create the tranining and validation loop. This loop will be similar to the previous train/validation loops, however there are a few extra parameters needed due to the transformer architecture. 

You need to use the `attention_mask` and get the loss from the output of the model with `loss = output[0]`

In [14]:
# Hyperparameters
num_epochs = 3
learning_rate = 2e-4

# Get the compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create data loaders
train_loader = DataLoader(train_dataset, shuffle=True,
                          batch_size=16, drop_last=True)
validation_loader = DataLoader(val_dataset, batch_size=8,
                               drop_last=True)

# Setup the optimizer (AdamW converges much faster than SGD for transformer models)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model = model.to(device)

for epoch in range(num_epochs):
    
    train_loss, val_loss, train_acc, valid_acc = 0., 0., 0., 0.
    
    start = time.time()
    # Training loop starts
    model.train() # put the model in training mode
    for batch in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()
        # Put data, label and attention mask to the correct device
        data = batch["input_ids"].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch["labels"].to(device)
        
        # Make forward pass
        output = model(data, attention_mask=attention_mask, labels=label)
        
        # Calculate the loss (this comes from the output)
        loss = output[0]
        # Make backwards pass (calculate gradients)
        loss.backward()
        # Accumulate training accuracy and loss
        train_acc += calculate_accuracy(output.logits, label).item()
        train_loss += loss.item()
        # Update weights
        optimizer.step()
    
    # Validation loop:
    # This loop tests the trained network on validation dataset
    # No weight updates here
    # torch.no_grad() reduces memory usage when not training the network
    model.eval() # Activate evaluation mode
    with torch.no_grad():
        for batch in validation_loader:
            data = batch["input_ids"].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label = batch["labels"].to(device)
            # Make forward pass with the trained model so far
            output = model(data, attention_mask=attention_mask, labels=label)
            # Accumulate validation accuracy and loss
            valid_acc += calculate_accuracy(output.logits, label).item()
            val_loss += output[0].item()
        
    # Take averages
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(validation_loader)
    valid_acc /= len(validation_loader)
    
    end = time.time()
    
    print("Epoch %d: train loss %.3f, train acc %.3f, val loss %.3f, val acc %.3f, seconds % .3f " % (
        epoch+1, train_loss, train_acc, val_loss, valid_acc, end-start))

Epoch 1: train loss 0.523, train acc 0.754, val loss 0.413, val acc 0.840, seconds  26.572 


Epoch 2: train loss 0.401, train acc 0.815, val loss 0.330, val acc 0.895, seconds  26.730 


Epoch 3: train loss 0.360, train acc 0.838, val loss 0.301, val acc 0.885, seconds  27.060 


<!-- Subsection Header -->
<div id="section3-1" style="border-left: 3px solid #4C32E2; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #4C32E2;">3.1 Looking at what's going on</h3>
</div>

The fine-tuned BERT model is able to correctly classify the sentiment of the most of the records in the validation set. Let's observe in more detail how the sentences are tokenized and encoded. You can do this by picking one sentence as example to look at.

In [15]:
st = val_texts[19]
print(f"Sentence: {st}")
tok = tokenizer(st, truncation=True, padding=True)
print(f"Encoded Sentence: {tok['input_ids']}")

Sentence: An excellent resource for all scanner owners.  Seems to be up to date and all inclusive.  I highly recommend this product!
Encoded Sentence: [101, 2019, 6581, 7692, 2005, 2035, 26221, 5608, 1012, 3849, 2000, 2022, 2039, 2000, 3058, 1998, 2035, 18678, 1012, 1045, 3811, 16755, 2023, 4031, 999, 102]


Print the vocabulary size.

In [16]:
# The mapped vocabulary is stored in tokenizer.vocab
tokenizer.vocab_size

30522

Use the encoded sentence with the tokenizer to recover the original sentence. 

In [17]:
# Methods convert_ids_to_tokens and convert_tokens_to_ids allow to see how sentences are tokenized
print(tokenizer.convert_ids_to_tokens(tok["input_ids"]))

['[CLS]', 'an', 'excellent', 'resource', 'for', 'all', 'scanner', 'owners', '.', 'seems', 'to', 'be', 'up', 'to', 'date', 'and', 'all', 'inclusive', '.', 'i', 'highly', 'recommend', 'this', 'product', '!', '[SEP]']


<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">4. Getting predictions on the test data</h2>
</div>

After the model is trained, you can focus on getting test data to make predictions with. Do this by:
- Reading and format the test dataset
- Passing the test data to your trained model and make predictions

In [18]:
# Read the test data (It doesn't have the isPositive label)
df_test = pd.read_csv("../data/NLP-REVIEW-DATA-CLASSIFICATION-TEST.csv")
df_test.head()

,ID,reviewText,summary,verified,time,log_votes
0,33276,I've been using greeting card software for wel...,Absolutely awful.,False,1300233600,0.000000
1,20859,"This version worked well for me, have upgraded...",Good for virtual machine on a mac,True,1448755200,0.000000
2,63500,Great!,Five Stars,True,1456963200,0.000000
3,4950,I can assure you that any five star review was...,SCAM,False,1400803200,2.197225
4,26509,Overall the product really seems the same but ...,Has potential but many glitches and really the...,False,1419206400,0.000000


Just as before, drop the rows that don't have the __reviewText__.

In [19]:
df_test.dropna(subset=["reviewText"], inplace=True)

Making predictions will also take a long time with this model. To get results quickly, start by only making predictions with 15 datapoints from the test set.

In [20]:
test_texts = df_test["reviewText"].tolist()[0:15]

In [21]:
test_encodings = tokenizer(test_texts,
                           truncation=True,
                           padding=True)

Create labels for the test dataset to pass zeros using `[0]*len(test_texts)`.

In [22]:
test_dataset = ReviewDataset(test_encodings, [0]*len(test_texts))

Then, create a dataloader for the test set and record the corresponding predictions.

In [23]:
test_loader = DataLoader(test_dataset, batch_size=4)
test_predictions = []
model.eval()

with torch.no_grad():
    for batch in test_loader:
        data = batch["input_ids"].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch["labels"].to(device)
        output = model(data, attention_mask=attention_mask, labels=label)
        predictions = torch.argmax(output.logits, dim=-1)
        test_predictions.extend(predictions.cpu().numpy())

Finally, pick an example sentence and examine the prediction. Does the prediction look correct? 

Remember 

- 1->positive class 
- 0->negative class

In [24]:
k = 13
print(f'Text: {test_texts[k]}')
print(f'Prediction: {test_predictions[k]}')

Text: I have been using this product for five or six years.  This purchase was my annual subscription renewal.  It has the features that I need, and seems to protect the three PC's that we have while using the internet.  I will be looking at the mobile apps for a tablet and smart phone.
Prediction: 1


<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Improve Model Performance</h4>
        <p>You trained the model for 3 epochs. Would you get better results from the validation dataset if the model trained longer?</p>
        <p><strong>Your task:</strong></p>
        <ol>
            <li>Make a note of your last <code>Val_loss</code> result.</li>
            <li>In the <a href="#section3">Training and testing the model</a> section, change the <code>num_epochs</code> parameter to <code>5</code>.</li>
            <li>Re-run the code blocks to load the pre-trained model, and train your model.</li>
            <li>Did <code>Val_loss</code> improve?</li>
        </ol>
    </div>
</div>

<div style="background-color: #f2f0fc; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2f1381; border-bottom: 1px solid #2f1381; padding-bottom: 5px;">Conclusion</h3>
    <p style="color: #2f1381;">In this lab, you have:</p>
    <ul>
        <li style="color: #2f1381;">Learned how to import a pre-trained Transformer model</li>
        <li style="color: #2f1381;">Fine-tuned a BERT model for sentiment classification</li>
        <li style="color: #2f1381;">Made predictions on new data using your fine-tuned model</li>
    </ul>
    <p style="color: #2f1381;">Although you used a lighter version of the BERT model, these types of models tend to use large amounts of compute power. For that reason, you only worked with the first 2000 datapoints of the dataset. To see more general results, you need to spend more time training while using the whole dataset.</p>
    <h4 style="color: #2f1381; margin-top: 15px;">Next Steps</h4>
    <p style="color: #2f1381;">In the next lab, you will learn how to read images and plot them as you start to learn about computer vision problems.</p>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../common/images/mlu-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!